<link rel="stylesheet" href="notebooks/styles.css">

<div class="title-wrap">
  <h1 class="title-main" style="font-weight: bold; font-size: 2.65rem; margin-bottom: 0.5rem;">
  Spatial Data Science Approaches to Wildfire Severity Modeling
</h1>
<h2 class="title-sub" style="font-style: italic; font-size: 1.8rem; margin-top: 0rem; margin-bottom: 0.2rem;">
  A GIS‑Driven, Tree‑Based Machine Learning Analysis of California Wildfires
</h2>
</div>

# Module 10B: *Fire Damage Feature Ablation*
##### Version Number: 4.0
---
### Contents  
> *Build Models*\
> *SHAP*\
> *Set Ablation*
---
### Notes
---
### Inputs
---
### Outputs 
---
### User Defined Dependencies

In [1]:
import os
import sys

# Allow import of custom modules from the parent directory
sys.path.append(os.path.abspath(os.path.join('..')))

from src.data_utils import *
from src.model_utils import *
from src.plot_utils import *

---
### Third Party Dependencies

In [2]:
# Core libraries
import pandas as pd
import numpy as np
import json

# Modeling libraries
from sklearn.ensemble import RandomForestClassifier
import xgboost as xgb
from sklearn.model_selection import train_test_split

---

###  Load Data

In [3]:
X_damage = pd.read_csv('../data/processed/X_damage.csv')
y_damage = pd.read_csv('../data/processed/y_damage.csv').squeeze()  # Load as Series
details_damage = pd.read_csv('../data/processed/details_damage.csv')

pal_details = pd.read_csv('../data/processed/pal_details.csv')
pal_X = pd.read_csv('../data/processed/pal_X.csv')
pal_y = pd.read_csv('../data/processed/pal_y.csv')

best_strategy = pd.read_csv('../data/processed/damage_best_strategy.csv')

with open('model_parameters_ignition.json', 'r') as f:
    model_parameters = json.load(f)

with open('feature_sets.json', 'r') as f:
    feature_sets = json.load(f)

## Subset

In [ ]:
reform = pd.concat([X_damage,y_damage], axis=1)
subset = subset_df(reform, 'Target_Damage', 500)

y = subset['Target_Damage']
X = subset.drop(columns='Target_Damage')

In [5]:
X_rf, y_rf = apply_balancing('RF', best_strategy, X, y)
X_xgb, y_xgb = apply_balancing('XGB', best_strategy, X, y)

## Build Models

In [6]:
RF_parameters = model_parameters['Random Forest']
XGB_parameters = model_parameters['XGBoost']

# Build Final tuned models
damage_xgb = xgb.XGBClassifier(**XGB_parameters)
damage_rf = RandomForestClassifier(**RF_parameters)

display(RF_parameters)
display(XGB_parameters)

{'n_estimators': 50,
 'max_depth': 10,
 'min_samples_split': 5,
 'max_features': 'log2',
 'class_weight': 'balanced'}

{'objective': 'multi:softmax',
 'num_class': 3,
 'n_estimators': 150,
 'max_depth': 5,
 'learning_rate': 0.1,
 'verbosity': 0}

In [7]:
models = {
    "XGB": damage_xgb, 
    "RF": damage_rf
}

## SHAP

### XGB Damage SHAP

In [8]:
xgb_top_10 = get_shap(damage_xgb, X_xgb, y_xgb)
xgb_top_10



 23%|=====               | 582/2500 [00:11<00:36]       


 26%|=====               | 641/2500 [00:12<00:34]       


 28%|======              | 700/2500 [00:13<00:33]       


 30%|======              | 757/2500 [00:14<00:32]       


 33%|=======             | 815/2500 [00:15<00:31]       

...


 91%|==================  | 2274/2500 [00:41<00:04]       


 93%|=================== | 2334/2500 [00:42<00:02]       


 96%|=================== | 2394/2500 [00:43<00:01]       


 98%|===================| 2453/2500 [00:44<00:00]       

,0,1
0,elevation_range,0.644862
1,1000-hour Dead Fuel Moisture,0.503419
2,Palmer Drought Severity Index,0.458448
3,intermix_zone,0.446901
4,northness_mean_x_Daily Maximum Air Temperature,0.398868
5,Solar Radiation 7 Day Avg,0.382550
6,slope_std,0.361423
7,elevation_mean,0.356459
8,slope_mean,0.337324
9,SPEI 90-Day,0.325681


### RF Damage SHAP

In [9]:
rf_top_10 = get_shap(damage_rf, X_rf, y_rf)
rf_top_10 



 89%|==================  | 2227/2500 [00:11<00:01]       


 98%|===================| 2444/2500 [00:12<00:00]       

,0,1
0,influence_zone,0.022330
1,1000-hour Dead Fuel Moisture,0.015852
2,Solar Radiation 7 Day Avg,0.014167
3,Palmer Drought Severity Index,0.013531
4,Daily Minimum Air Temperature 7 Day Avg,0.012113
5,dominant_section_percent,0.011122
6,intermix_zone,0.010003
7,SPEI 90-Day,0.009877
8,Season_Summer,0.009350
9,slope_mean,0.009337


## Set Ablation

In [10]:
count = 0

for set_name, columns in feature_sets.items():
    count = count + len(columns)
    print(f"{set_name}: {columns}")
    print("\n")

Water Demand: ['Actual Evapotranspiration', 'Solar Radiation', 'Solar Radiation 7 Day Avg', 'Daily Minimum Air Temperature', 'Daily Maximum Air Temperature', 'Daily Maximum Air Temperature 7 Day Avg', 'Daily Minimum Air Temperature 7 Day Avg', 'Vapor Pressure Deficit', 'Vapor Pressure Deficit 7 Day Avg', 'Wind Speed', 'Wind Speed 7 Day Avg']


Water Supply: ['Precipitation', 'Precipitation 7 Day Avg', 'Maximum Relative Humidity', 'Minimum Relative Humidity', 'Maximum Relative Humidity 7 Day Avg', 'Minimum Relative Humidity 7 Day Avg', 'Specific Humidity', '100-hour Dead Fuel Moisture', '1000-hour Dead Fuel Moisture']


Water Supply Indexes: ['SPI 30-Day', 'SPI 180-Day', 'SPEI 30-Day', 'SPEI 90-Day', 'SPEI 180-Day', 'Palmer Drought Severity Index']


Fire Danger: ['Burning Index', 'Energy Release Component', 'Santa_Ana_Score']


Social: ['total_housing', 'total_population', 'housing_density', 'population_density', 'median_income']


Infrastructure: ['power_line_meters', 'power_line_dens

In [11]:
compare = []

compare.append(compare_model(damage_xgb, X, y, best_strategy,
                                     name = 'XGB', test_set = 'Full'))

compare.append(compare_model(damage_rf, X, y, best_strategy,
                                     name = 'RF', test_set = 'Full'))

for set_name, set_list in feature_sets.items():
    for model_name, model in models.items():
        print("Testing " + f"{model_name}: {set_name}")
        X_section = X.drop(columns = set_list)
        compare.append(compare_model(model, X_section, y, best_strategy,
                                     name = model_name, test_set = set_name))

Testing XGB: Water Demand


Testing RF: Water Demand


Testing XGB: Water Supply


Testing RF: Water Supply


Testing XGB: Water Supply Indexes


...

Testing RF: Others


Testing XGB: Coded Ecoregions


Testing RF: Coded Ecoregions


Testing XGB: Coded Seasons


Testing RF: Coded Seasons


In [12]:
comparisons = pd.DataFrame(compare)

In [13]:
XGB = comparisons[comparisons['Model'] == 'XGB'] 
RF = comparisons[comparisons['Model'] == 'RF'] 

In [14]:
full_XGB = XGB.loc[XGB['Test Set']=='Full','Macro F1'].values[0]
full_RF = RF.loc[RF['Test Set']=='Full','Macro F1'].values[0]

In [15]:
XGB['Macro F1 Percent Difference'] = ((full_XGB - XGB['Macro F1']) / full_XGB) * 100
RF['Macro F1 Percent Difference'] = ((full_RF - RF['Macro F1']) / full_RF) * 100

C:\Users\dusti\AppData\Local\Temp\ipykernel_9500\2524239291.py:1: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  XGB['Macro F1 Percent Difference'] = ((full_XGB - XGB['Macro F1']) / full_XGB) * 100
C:\Users\dusti\AppData\Local\Temp\ipykernel_9500\2524239291.py:2: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  RF['Macro F1 Percent Difference'] = ((full_RF - RF['Macro F1']) / full_RF) * 100


In [16]:
RF

,Test Set,Model,Weighted F1,Macro F1,High Risk Recall,Macro F1 Percent Difference
1,Full,RF,0.846541,0.843774,0.916667,0.000000
3,Water Demand,RF,0.857057,0.853201,0.885417,-1.117274
5,Water Supply,RF,0.860932,0.858265,0.916667,-1.717450
7,Water Supply Indexes,RF,0.799252,0.793870,0.906250,5.914363
9,Fire Danger,RF,0.845015,0.841486,0.906250,0.271199
11,Social,RF,0.858817,0.855461,0.916667,-1.385080
13,Infrastructure,RF,0.844907,0.841158,0.906250,0.309987
15,Elevation,RF,0.857142,0.853837,0.885417,-1.192609
17,WUI,RF,0.864537,0.862512,0.937500,-2.220768
19,Ecoregion,RF,0.850925,0.849123,0.906250,-0.633905


In [17]:
XGB

,Test Set,Model,Weighted F1,Macro F1,High Risk Recall,Macro F1 Percent Difference
0,Full,XGB,0.925513,0.926553,0.968750,0.000000
2,Water Demand,XGB,0.925406,0.926241,0.968750,0.033718
4,Water Supply,XGB,0.919202,0.920327,0.968750,0.671966
6,Water Supply Indexes,XGB,0.842428,0.839732,0.906250,9.370364
8,Fire Danger,XGB,0.917842,0.917652,0.947917,0.960701
10,Social,XGB,0.923470,0.924069,0.968750,0.268099
12,Infrastructure,XGB,0.921267,0.922340,0.958333,0.454664
14,Elevation,XGB,0.923589,0.923952,0.947917,0.280747
16,WUI,XGB,0.925622,0.926201,0.947917,0.037992
18,Ecoregion,XGB,0.917610,0.918084,0.937500,0.914055


In [18]:
len(X.columns)

113

In [19]:
count

113